# 🏗️ Notebook 1: Reminder / Alert — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/reminder-alert
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A system that fires reminders at specified times — push, email, SMS. Think Google Calendar, medication reminders, anniversary alerts.

Challenge: **billions of scheduled firings** with tight time accuracy (±1s).

## Requirements

### Functional
- Schedule one-off or recurring reminders.
- Cancel/reschedule.
- Deliver via chosen channel.
- User sees delivery status.

### Non-functional
- At-least-once delivery; receiver must be idempotent.
- ±1s firing accuracy.
- Survive worker crashes.

## Back-of-envelope

- 1B scheduled reminders.
- Firings peaking at 100k/s around top-of-hour.
- Storage: `(user_id, fire_at, payload)` ≈ 200 B × 1B = 200 GB — fits in a sharded SQL.

## High-level architecture

```
  [API] ──► scheduler DB (indexed by fire_at)
                │
                ▼
  dispatcher workers (poll next N ready rows)
                │
                ▼
        delivery queue (per channel)
                │
   ┌────────────┼────────────┐
   ▼            ▼            ▼
 Push         Email         SMS
```

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.